# Arabic Cursive OCR Benchmark — Analysis

Compare runs and browse the worst errors. Works in Colab or locally.
Point `RESULTS` at a folder that holds one or more run directories
(e.g. `/content/drive/MyDrive/arabic-ocr-bench/results` on Colab, or `results` locally).

In [ ]:
# Colab only: mount Drive to read saved runs. Skip if running locally.
try:
    from google.colab import drive; drive.mount('/content/drive')
    RESULTS = '/content/drive/MyDrive/arabic-ocr-bench/results'
except Exception:
    RESULTS = 'results'
print('results dir:', RESULTS)

## 1. Leaderboard across all runs

In [ ]:
import json, pandas as pd, pathlib
rows = []
for sc in pathlib.Path(RESULTS).glob('*/scores.jsonl'):
    run = sc.parent.name
    recs = [json.loads(l) for l in sc.read_text().splitlines() if l.strip()]
    d = pd.DataFrame(recs)
    for (model, ds), g in d.groupby(['model','dataset']):
        rows.append({
            'run': run, 'model': model, 'dataset': ds, 'n': len(g),
            'cer_light': round(g['cer_light'].mean(), 4),
            'cer_raw': round(g['cer_raw'].mean(), 4),
            'wer_light': round(g['wer_light'].mean(), 4),
            'exact_light': round(g['exact_light'].mean(), 4),
            'diac_f1': round(g['diac_f1'].mean(), 4),
            'latency_ms': round(g['latency_ms'].mean(), 1),
        })
board = pd.DataFrame(rows).sort_values(['dataset','cer_light'])
board

## 2. CER distribution for one run/model

In [ ]:
RUN = board.iloc[0]['run']; MODEL = board.iloc[0]['model']
recs = [json.loads(l) for l in (pathlib.Path(RESULTS)/RUN/'scores.jsonl').read_text().splitlines() if l.strip()]
s = pd.DataFrame([r for r in recs if r['model']==MODEL])
ax = s['cer_light'].plot(kind='hist', bins=20, title=f'{MODEL} — CER (light) per line')
ax.set_xlabel('CER'); ax

## 3. Worst errors — image + ground truth + prediction

Re-loads the line images by row index (needs internet the first time).
The dataset is read from the run's sample ids (`<dataset>/<split>/<idx>`).

In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
_HF = {'muharaf_lines': ('aamijar/muharaf-public', 'test'),
       'kitab_khatt': ('ahmedheakl/arocrbench_khatt', 'train'),
       'kitab_muharaf': ('ahmedheakl/arocrbench_muharaf', 'train'),
       'kitab_historicalbooks': ('ahmedheakl/arocrbench_historicalbooks', 'train'),
       'kitab_hindawi': ('ahmedheakl/arocrbench_hindawi', 'train')}
_dsname = s.iloc[0]['sample_id'].split('/')[0] if 'sample_id' in s else 'muharaf_lines'
_hf_id, _split = _HF.get(_dsname, ('aamijar/muharaf-public', 'test'))
ds = load_dataset(_hf_id, split=_split)

preds = [json.loads(l) for l in (pathlib.Path(RESULTS)/RUN/'predictions.jsonl').read_text().splitlines() if l.strip()]
pmap = {p['sample_id']: p for p in preds if p['model']==MODEL}
worst = s.sort_values('cer_light', ascending=False).head(8)

for _, r in worst.iterrows():
    row_idx = int(r['sample_id'].split('/')[-1])
    p = pmap[r['sample_id']]
    plt.figure(figsize=(12, 2)); plt.imshow(ds[row_idx]['image']); plt.axis('off')
    plt.title(f"CER {r['cer_light']:.2f}"); plt.show()
    print('GT  :', p['ground_truth'])
    print('PRED:', p['cleaned_text'] or '(empty / error: %s)' % p.get('error'))
    print('-' * 80)